In [1]:
import hoda
import tensorly as tl
random_state=42

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation



paradigm = P300(resample=48)
dataset = BNCI2014008()
epochs, labels, meta = paradigm.get_data(
    dataset=dataset, 
    subjects=[5],
    return_epochs=True
)

<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(
BNCI2014008 has been renamed to BNCI2014_008. BNCI2014008 will be removed in version 1.1.
The dataset class name 'BNCI2014008' must be an abb

To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.
Adding metadata with 3 columns
Adding metadata with 3 columns
4200 matching events found
No baseline correction applied


/usr/local/lib/python3.10/dist-packages/moabb/paradigms/base.py:354: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  X = mne.concatenate_epochs(X)


In [3]:
hoda_params = dict(
    max_iter=128,
    tol=1e-6,
    init ='random',
    shrinkage='lw',
    toeplitz=None,
    obj='rt',
    solver='lanczos',
    taper=False,
    extra_train_info=False,
    verbose=False, 
    random_state=random_state,
    ortho=False
)

In [4]:

from hoda.hoda import HODA,BTTDA
from sklearn.model_selection import GridSearchCV, cross_val_score
from hoda.tensorize import Tensorize, Vectorize
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd

X = tl.tensor(epochs.get_data())
y = labels
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
"""
hoda = Pipeline([
    ('tensor', Tensorize(method=None)),
    ('clf', GridSearchCV(
        Pipeline([
            ('hoda', HODA(**hoda_params, delta=None,)),
            ('vec', Vectorize()),
            ('zscore', StandardScaler()),
            ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')),
        ]),
        param_grid=dict(hoda__rank=[1,2,4,8,16,32]),
        n_jobs=4,
        cv=cv,
    )),
])
res_hoda = cross_val_score(hoda, X,y, cv=cv, n_jobs=4)
res_hoda = pd.DataFrame(res_hoda)
res_hoda
"""

/tmp/ipykernel_8671/1392448220.py:10: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  X = tl.tensor(epochs.get_data())


"\nhoda = Pipeline([\n    ('tensor', Tensorize(method=None)),\n    ('clf', GridSearchCV(\n        Pipeline([\n            ('hoda', HODA(**hoda_params, delta=None,)),\n            ('vec', Vectorize()),\n            ('zscore', StandardScaler()),\n            ('lda', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')),\n        ]),\n        param_grid=dict(hoda__rank=[1,2,4,8,16,32]),\n        n_jobs=4,\n        cv=cv,\n    )),\n])\nres_hoda = cross_val_score(hoda, X,y, cv=cv, n_jobs=4)\nres_hoda = pd.DataFrame(res_hoda)\nres_hoda\n"

In [5]:
from hoda.hoda import BTTDA, InfoBTTDA, GreedyBTTDA

max_blocks = 8
pipeline = Pipeline([
    ('bttda', GreedyBTTDA(
        ranks=[None]*max_blocks,
        hoda_params=dict(
            **hoda_params,
            delta=None,
        ),
        extra_train_info=False,
        verbose=False
    )),
    ('vec', Vectorize()),
    ('zscore', StandardScaler()),
    #('select', SelectF(alpha=1)),
    ('clf', LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'))
])

In [ ]:
result = cross_validate(pipeline,X,y, cv=cv, return_estimator=True, return_indices=True)

In [ ]:
import math
from sklearn.metrics import roc_auc_score
import pandas as pd

block_results = []
for fold in range(cv.n_splits):
    estimator = result['estimator'][fold]
    train_idc = result['indices']['train'][fold]
    test_idc = result['indices']['test'][fold]
    Xt = estimator[:2].transform(X)
    for n_blocks in range(1, max_blocks+1):
        block_ranks = estimator[0].ranks_[:n_blocks]        
        n_features = sum([math.prod(ml_rank) for ml_rank in block_ranks])
        pipeline[2:].fit(Xt[train_idc,:n_features],y[train_idc])
        roc_auc = roc_auc_score(y[test_idc],pipeline[2:].decision_function(Xt[test_idc,:n_features]),)
        block_results.append(dict(fold=fold, n_blocks=n_blocks,roc_auc=roc_auc, n_features=n_features))
block_results = pd.DataFrame(block_results)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('default')
sns.lineplot(data=block_results, x='n_blocks', y='roc_auc')
#plt.axhline(res_hoda[0].mean(), color='red')

In [ ]:
"""
max_features = X[0].size
for i in range(math.ceil(math.sqrt(max_features/max_blocks))):
    n = max_blocks*(i**2)
    plt.plot([0, max_blocks], [0, n], color='lightgray')
plt.axhline(max_features, color='red')
plt.axhline(min(X[0].shape)**2, color='green')

sns.lineplot(data=block_results, x='n_blocks', y='n_features')
"""